# ERA5 bulk ingestion

Set the region, variables, and UTC date interval below, then run the three code cells in order. `run_bulk` reads the public ERA5 Zarr store in parallel up to `WORKERS`, skips complete lake days, and writes each finished daily partition atomically. Restart the same interval after interruption; completed days are skipped automatically. Install the repository's `.[weather,notebooks]` extras in the selected kernel first.

In [1]:
from datetime import date

REGION_NAME = "SoCal"
NORTH = 35.0
WEST = -121.0
SOUTH = 32.5
EAST = -114.0

START_DATE = date(2021,1, 1)  # Inclusive; resume after the last completed date.
END_DATE = date(2026, 10, 1)    # Exclusive.
VARIABLES = ("2m_temperature",)
WORKERS = 4
REFRESH = False                # True re-fetches even complete partitions.
LAKE_ROOT = None               # None uses <project>/data/lake/weather/era5.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/comm_research").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the commodity-research checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from comm_research.markets.weather.models.grid import WeatherGrid
from comm_research.markets.weather.scrapers.era5.regional import (
    DEFAULT_LAKE_ROOT, ERA5Scraper,
)

In [3]:
grid = WeatherGrid(north=NORTH, west=WEST, south=SOUTH, east=EAST, name=REGION_NAME)
scraper = ERA5Scraper(
    grid, 
    variables=VARIABLES,
    lake_root=LAKE_ROOT or DEFAULT_LAKE_ROOT,
    workers=WORKERS,
)

summary = scraper.run_bulk(START_DATE, END_DATE, refresh=REFRESH)
print(summary)

RunSummary(downloaded_requests=1, written_days=1134, skipped_days=965, received_rows=8681904)
